# Sabaic OCR — Synthetic v2 Resume-Safe Training
Dedicated Colab for the corrected classifier training. It is designed to survive Colab/network disconnects:
- `best.pt` and `last.pt` live on Google Drive.
- Synthetic dataset is backed up once to a Drive `.tar` archive and restored automatically after a runtime reset.
- `retrain_synthetic_v2.py` automatically resumes from `checkpoints/synthetic_v2/last.pt`.
- The baseline `checkpoints/synthetic/best.pt` is never overwritten.


In [ ]:
import os, subprocess
from pathlib import Path

assert os.path.exists('/content'), 'This notebook is intended for Google Colab.'
repo = Path('/content/Sabaic-OCR-YOLO')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/7eaur/Sabaic-OCR-YOLO.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)

os.chdir(repo)
subprocess.run(['pip','install','-r','requirements.txt','-q'], check=True)
subprocess.run(['pip','install','-e','.', '--no-deps','--no-build-isolation','-q'], check=True)
subprocess.run(['python','scripts/check_environment.py'], check=True)
subprocess.run(['pytest','-q'], check=True)


## 1) Mount Google Drive and make checkpoints persistent
Run this once per Colab runtime. Every completed epoch writes `last.pt` to Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Sabaic-OCR-YOLO')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

os.chdir('/content/Sabaic-OCR-YOLO')
local_ckpt = Path('checkpoints')
if local_ckpt.is_symlink():
    local_ckpt.unlink()
elif local_ckpt.exists():
    shutil.rmtree(local_ckpt)
local_ckpt.symlink_to(DRIVE_CKPT, target_is_directory=True)

print('Baseline best:', Path('checkpoints/synthetic/best.pt').exists())
print('V2 last (resume point):', Path('checkpoints/synthetic_v2/last.pt').exists())
print('V2 best:', Path('checkpoints/synthetic_v2/best.pt').exists())
print('Persistent checkpoint root:', local_ckpt.resolve())


## 2) Persist the project font privately in your Drive
The font is **not committed to GitHub**. On the first run, upload the TTF once; the notebook stores a private copy in your own Drive for later reconnects.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os, subprocess

os.chdir('/content/Sabaic-OCR-YOLO')
local_font = Path('assets/fonts/NotoSansOldSouthArabian-Regular.ttf')
drive_font = DRIVE_ROOT / 'private_inputs/NotoSansOldSouthArabian-Regular.ttf'
local_font.parent.mkdir(parents=True, exist_ok=True)
drive_font.parent.mkdir(parents=True, exist_ok=True)

if drive_font.exists():
    shutil.copy2(drive_font, local_font)
    print('Restored font from your Drive.')
else:
    uploaded = files.upload()
    ttf_items = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.ttf')]
    if not ttf_items:
        raise RuntimeError('No .ttf font was uploaded.')
    local_font.write_bytes(ttf_items[0][1])
    shutil.copy2(local_font, drive_font)
    print('Font saved privately to your Drive for future reconnects.')

subprocess.run(['python','scripts/validate_font.py'], check=True)


## 3) Restore or generate the reviewed 5,000/500/100 synthetic dataset
The first run generates and audits the dataset, then creates a Drive archive. After a future runtime reset, this cell restores the exact dataset instead of regenerating it.


In [ ]:
import os, tarfile, shutil, subprocess
from pathlib import Path

os.chdir('/content/Sabaic-OCR-YOLO')
dataset_root = Path('data/synthetic')
archive = DRIVE_ROOT / 'datasets/synthetic_seed42_5000_500_100.tar'
archive.parent.mkdir(parents=True, exist_ok=True)

def split_count(split):
    p = dataset_root / 'images' / split
    return len(list(p.glob('*.jpg'))) if p.exists() else 0

counts = {s: split_count(s) for s in ('train','val','test')}
print('Local counts before restore:', counts)

if counts != {'train':5000,'val':500,'test':100}:
    if dataset_root.exists():
        shutil.rmtree(dataset_root)
    if archive.exists():
        print('Restoring synthetic dataset from Drive archive...')
        with tarfile.open(archive, 'r') as tf:
            tf.extractall(path='data')
    else:
        print('No archive found. Generating reviewed synthetic dataset...')
        subprocess.run([
            'python','scripts/generate_synthetic.py',
            '--output','data/synthetic',
            '--train','5000','--val','500','--test','100',
            '--seed','42','--overwrite'
        ], check=True)
        subprocess.run([
            'python','scripts/review_synthetic_stage.py',
            '--require-train','5000','--require-val','500','--require-test','100'
        ], check=True)
        print('Backing up dataset to Drive. This is done once...')
        with tarfile.open(archive, 'w') as tf:
            tf.add(dataset_root, arcname='synthetic')

counts = {s: split_count(s) for s in ('train','val','test')}
print('Local counts after restore/generation:', counts)
assert counts == {'train':5000,'val':500,'test':100}


## 4) Mandatory audit before v2 training
Do not train if this gate fails.


In [ ]:
import os, subprocess
os.chdir('/content/Sabaic-OCR-YOLO')
subprocess.run([
    'python','scripts/review_synthetic_stage.py',
    '--require-train','5000','--require-val','500','--require-test','100'
], check=True)


## 5) Corrective Synthetic v2 training — automatic resume
Run this cell whenever you want to train.

- First run: starts from `checkpoints/synthetic/best.pt`, preserves localization, and resets only classification logits.
- After any disconnect: rerun cells 0–4, then this cell. It automatically resumes from `checkpoints/synthetic_v2/last.pt`.
- `last.pt` is saved after **every completed epoch** to Google Drive.


In [ ]:
import os, subprocess, torch
from pathlib import Path

os.chdir('/content/Sabaic-OCR-YOLO')
assert torch.cuda.is_available(), 'Enable a GPU runtime (T4 or better) before training.'
assert Path('checkpoints/synthetic/best.pt').exists(), 'Baseline best.pt is missing from Google Drive.'

log_dir = DRIVE_ROOT / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / 'synthetic_v2_training.log'

cmd = "set -o pipefail; python scripts/retrain_synthetic_v2.py --config config/train_synthetic_v2.json 2>&1 | tee -a '{}'".format(log_path)
result = subprocess.run(['bash','-lc',cmd])
if result.returncode != 0:
    raise RuntimeError(f'v2 training failed with exit code {result.returncode}')


## 6) Verify training state
This cell is safe to run after training or after a reconnect.


In [ ]:
from pathlib import Path
import torch, os

os.chdir('/content/Sabaic-OCR-YOLO')
for name in ['checkpoints/synthetic_v2/last.pt','checkpoints/synthetic_v2/best.pt']:
    p = Path(name)
    print(name, 'exists=', p.exists(), 'size_MB=', round(p.stat().st_size/1024**2,2) if p.exists() else None)
    if p.exists():
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        print('  completed epoch:', int(ckpt['epoch']) + 1, '/ 30')
        print('  best val loss:', ckpt['best_metric'])


## 7) Evaluate v2 on the independent synthetic test split
Run only after v2 completes 30/30. Results are written directly to your Drive.


In [ ]:
import os, subprocess
from pathlib import Path

os.chdir('/content/Sabaic-OCR-YOLO')
v2_best = Path('checkpoints/synthetic_v2/best.pt')
assert v2_best.exists(), 'v2 best.pt not found.'

result_path = DRIVE_ROOT / 'results/synthetic_v2_test_metrics.json'
result_path.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python','scripts/evaluate.py',
    '--checkpoint',str(v2_best),
    '--images','data/synthetic/images/test',
    '--labels','data/synthetic/labels/test',
    '--transcripts','data/synthetic/transcripts/test',
    '--batch-size','8',
    '--output',str(result_path)
], check=True)

print('Saved evaluation:', result_path)
